# Lab: Building ReAct Agents with LangChain and LangGraph
#
# Objectives:
# 1) Build a simple tool-calling agent with LangChain
# 2) Understand the ReAct loop: think -> act -> observe -> answer
# 3) Build a simple graph-based agent with LangGraph
# 4) Compare LangChain vs LangGraph

# Part 1 — LangChain Agent

In this part, we build a simple agent using LangChain.

The agent combines:
- a language model
- tools
- instructions

The model can decide whether it should:
- answer directly
- or call a tool first

This is a simple form of ReAct-style behavior:
1. Reason about the task
2. Act by calling a tool
3. Observe the tool result
4. Produce the final answer

# Importing Necessary Libraries

Here we import the necessary libraries for both LangChain and LangGraph:

- **TypedDict** and **Annotated** are used to define the structure of the state for LangGraph.
- **LangChain's core components** help us create an agent and define tools it can call.
- **ChatGroq** wraps the Groq model for LangChain integration.
- **LangGraph components** such as **StateGraph** and **ToolNode** are used for creating graph-based workflows that give us more control over agent behavior.

pip install langchain langgraph langchain-ollama langchain-groq numpy pandas matplotlib

In [5]:
import sys
!{sys.executable} -m pip install langchain langgraph langchain-groq

  Using cached langchain-1.2.15-py3-none-any.whl.metadata (5.8 kB)
  Using cached langgraph-1.1.6-py3-none-any.whl.metadata (8.0 kB)
  Using cached langgraph_checkpoint-4.0.2-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_prebuilt-1.0.9-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.3.13-py3-none-any.whl.metadata (1.6 kB)
Using cached langchain-1.2.15-py3-none-any.whl (112 kB)
Using cached langgraph-1.1.6-py3-none-any.whl (169 kB)
Using cached langgraph_checkpoint-4.0.2-py3-none-any.whl (51 kB)
Using cached langgraph_prebuilt-1.0.9-py3-none-any.whl (35 kB)
Using cached langgraph_sdk-0.3.13-py3-none-any.whl (96 kB)

   ------ --------------------------------- 1/6 [langgraph-sdk]
   ------ --------------------------------- 1/6 [langgraph-sdk]
   ------ --------------------------------- 1/6 [langgraph-sdk]
   ------ --------------------------------- 1/6 [langgraph-sdk]
   ------ --------------------------------- 1/6 [langgraph-sdk]
   ------ ---------------

In [6]:
from typing import TypedDict, Annotated

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# LangChain Model Setup

In this section, we initialize the **model** using **Ollama**. We're using the **qwen3:4b** model here, but you can replace it with any tool-compatible model you prefer.
This model will be used by the agent to generate responses based on user input.

We also set **temperature=0**, meaning the output will be deterministic (no randomness).

In [7]:
pip install langchain-ollama

  Using cached ollama-0.6.1-py3-none-any.whl.metadata (4.3 kB)
Using cached ollama-0.6.1-py3-none-any.whl (14 kB)

   ---------------------------------------- 0/2 [ollama]
   -------------------- ------------------- 1/2 [langchain-ollama]
   -------------------- ------------------- 1/2 [langchain-ollama]
   ---------------------------------------- 2/2 [langchain-ollama]

Note: you may need to restart the kernel to use updated packages.


In [8]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434",
    temperature=0
)

# Tool Definitions

Here, we define two tools for the agent:

1. **Calculator**: This tool takes a string representing a mathematical expression and evaluates it (e.g., '15 * 8').
2. **Course Lookup**: This tool returns short course information based on a topic (e.g., "LangChain" or "LangGraph").

The **@tool** decorator in LangChain tells the agent that these are callable functions the agent can invoke.

In [9]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a simple math expression like '25 * 4 + 10'."""
    try:
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"


@tool
def course_lookup(topic: str) -> str:
    """Return short course information for a requested topic."""
    data = {
        "langchain": "LangChain is a framework for building LLM applications with prompts, tools, chains, and agents.",
        "langgraph": "LangGraph is a framework for building stateful, graph-based agent workflows.",
        "react": "ReAct stands for Reason plus Act. The model reasons, uses tools, observes results, and then continues."
    }
    return data.get(topic.lower(), f"No course information found for '{topic}'.")

In [10]:
print(calculator.invoke("5 * 12 + 3"))
print(course_lookup.invoke("langgraph"))

Result: 63
LangGraph is a framework for building stateful, graph-based agent workflows.


In [11]:
agent = create_agent(
    model=model,
    tools=[calculator, course_lookup],
    system_prompt=(
        "You are a helpful study assistant. "
        "Use tools when needed. "
        "If the user asks about math, use the calculator tool. "
        "If the user asks about course topics, use the course_lookup tool."
    ),
)

In [12]:
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is 15 * 8, and what is LangGraph?"}
        ]
    }
)

response

{'messages': [HumanMessage(content='What is 15 * 8, and what is LangGraph?', additional_kwargs={}, response_metadata={}, id='f8cb7d34-1ab2-4efe-ad1b-aafd84a51835'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-04-16T16:42:35.895766Z', 'done': True, 'done_reason': 'stop', 'total_duration': 418863063800, 'load_duration': 19509490900, 'prompt_eval_count': 234, 'prompt_eval_duration': 67493449000, 'eval_count': 823, 'eval_duration': 329657228400, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019d9725-df84-7512-bc81-f31128fce10a-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '15 * 8'}, 'id': 'c2237558-12e9-4337-8e30-9f698342ea06', 'type': 'tool_call'}, {'name': 'course_lookup', 'args': {'topic': 'LangGraph'}, 'id': '455e6b0a-09a0-42c5-b774-ba915cb78727', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 234, 'output_tokens': 823, 'total_tokens': 1057

In [13]:
print(response["messages"][-1].content)  # Display the final answer

The result of $ 15 \times 8 $ is **120**.  

LangGraph is a framework for building stateful, graph-based agent workflows.


# LangGraph Setup

In this section, we define the **LangGraph** workflow.

1. **State**: The state represents the conversation history, which is stored as messages.
2. **Nodes**:
   - The **chatbot node** calls the model to get a response.
   - The **tool node** runs the tools (calculator and course lookup).
3. **Graph Flow**: The flow starts at the chatbot node. If the model needs tools, it calls the tool node, executes the tools, and then loops back to the chatbot node.

This is a **graph-based approach**, which gives us more control over the decision-making process of the agent.

# Importing Required Libraries for LangGraph

In this section, we import the necessary libraries for LangGraph to create a **graph-based agent**:

1. **`TypedDict` and `Annotated`**:
   - These are used to define the **structure** of the state in the **LangGraph**.
   - `TypedDict` allows us to specify that the state will be a dictionary with predefined keys, and `Annotated` is used to annotate that `messages` will hold the conversation history.

2. **`StateGraph`, `START`, `END`**:
   - **`StateGraph`** is the core class that allows us to define a graph of nodes (each node represents an action, such as calling the model or using a tool).
   - **`START`** and **`END`** represent the entry and exit points in the graph. The flow of the agent starts at `START` and ends at `END`.

3. **`add_messages`**:
   - This is a utility function to handle and manage **messages** (the conversation history). It allows us to add new messages to the state during the agent's workflow.

4. **`ToolNode` and `tools_condition`**:
   - **`ToolNode`** represents a node in the graph that executes **tools** (like our calculator or course lookup). It ensures that the tools are called when necessary.
   - **`tools_condition`** is a condition used to determine when to move from one node to another. In this case, it helps us decide when to execute a tool, based on the conversation flow.

In [14]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

In [15]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [16]:
tools = [calculator, course_lookup]
model_with_tools = model.bind_tools(tools)

In [17]:
def chatbot_node(state: AgentState):
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [18]:
tool_node = ToolNode(tools)

In [19]:
graph_builder = StateGraph(AgentState)

graph_builder.add_node("chatbot", chatbot_node)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition
)

graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile()

In [20]:
result = graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is 15 * 8, and what is LangGraph?"}
        ]
    }
)

In [21]:
for msg in result["messages"]:
    print(msg)
    print("-" * 80)

content='What is 15 * 8, and what is LangGraph?' additional_kwargs={} response_metadata={} id='f735511d-1372-444c-b3d1-50e13b66a236'
--------------------------------------------------------------------------------
content='' additional_kwargs={} response_metadata={'model': 'qwen3:4b', 'created_at': '2026-04-16T16:52:37.4538708Z', 'done': True, 'done_reason': 'stop', 'total_duration': 462657705800, 'load_duration': 553651200, 'prompt_eval_count': 196, 'prompt_eval_duration': 38412554900, 'eval_count': 1046, 'eval_duration': 422023830800, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'} id='lc_run--019d972e-6297-7a83-bd79-4fdba46aae44-0' tool_calls=[{'name': 'calculator', 'args': {'expression': '15 * 8'}, 'id': 'aeb5b2a3-cdaa-4d95-a1c5-f28eea488428', 'type': 'tool_call'}, {'name': 'course_lookup', 'args': {'topic': 'LangGraph'}, 'id': '83eaf0ec-42d6-4128-b734-24f403fddd02', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 196, 'output_tok

# Reflection Questions

**1) What is the difference between a normal LLM call and an agent?**

A normal LLM call is basically a one attempt style we send a prompt and the model replies. It has no memory of previous calls and can't actually do anything beyond returning an answer. An agent operates in a loop so the model can look at the situation and decide to call a tool (like a calculator), judge the reply and keep doing so until it has a correct answer. So the difference is that an agent can act and it's not just talking it's also interacting with other things.

**2) Why is ReAct useful?**

Reason + Act is useful because it forces the model to be explicit about its thinking before acting. Instead of guessing an answer or hallucinating, the model decides it needs to use a tool, calls it, gets the real result, and then answers based on that. This makes the agent much more reliable as we can follow exactly what happened at each step which also makes debugging easier.

**3) What is easier to use: LangChain or LangGraph?**

LangChain is easier to get started with as we call 'create_agent', pass in the model and tools and a system prompt and thats it. LangGraph requires definition of the state schema then create each node manually and connet edges and conditional transitions. 

**4) When would you prefer LangGraph over LangChain?**

I would choose LangGraph whenever the workflow gets complex enough that the high-level agent abstraction is causing problems. For example, multi-agent systems where different agents hand work to each other, pipelines that need human approval steps, or anything where we need to inspect state between steps. 

**5) What are the limitations of this simple agent?**

There is no memory so every invocation starts with no context from previous conversations. The tools are limited and the agent cannot call anything other than the two tools we defined. There is no protection against infinite loops, if the model keeps deciding to call tools without reaching a final answer, it could run forever. Very little error handling is used so a bad tool call or unexpected tool failure could ruin the flow. 

# Task: Create Your Own Graph with a Custom Chatbot Tool
# Objective:


# Your task is to create a graph-based workflow using LangGraph, with your own custom chatbot tool.

In [ ]:
from typing import TypedDict, Annotated
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

#Custom Tool: Movie Recommender (inspired from letterboxd)
@tool
def movie_recommender(genre: str) -> str:
    """Recommends a movie based on the requested genre."""
    recommendations = {
        "action": "Mad Max: Fury Road — non-stop, visually stunning, and one of the best action films ever made.",
        "comedy": "The Grand Budapest Hotel — witty, beautifully crafted, and genuinely funny.",
        "sci-fi": "Arrival — a thoughtful, mind-bending sci-fi film that is more about language than explosions.",
        "horror": "Hereditary — deeply unsettling and genuinely scary. Not for the faint-hearted.",
        "drama": "Parasite — gripping, unpredictable, and brilliantly written.",
        "animation": "Spider-Man: Into the Spider-Verse — visually unlike anything else, great story too.",
    }
    return recommendations.get(
        genre.lower(),
        f"No recommendation found for '{genre}'. Try: action, comedy, sci-fi, horror, drama, animation."
    )


#Custom Tool: Fun Fact Generator 
@tool
def fun_fact(topic: str) -> str:
    """Returns a fun fact about a given topic."""
    facts = {
        "space": "A day on Venus is longer than a year on Venus — it rotates so slowly that it completes an orbit around the Sun before finishing one full spin.",
        "animals": "Octopuses have three hearts, blue blood, and can edit their own RNA to adapt to temperature changes.",
        "history": "Cleopatra lived closer in time to the Moon landing than to the construction of the Great Pyramid of Giza.",
        "food": "Honey never spoils — archaeologists have found 3,000-year-old honey in Egyptian tombs that was still perfectly edible.",
        "math": "The number 40 is the only number whose letters appear in alphabetical order in English.",
    }
    return facts.get(
        topic.lower(),
        f"No fun fact found for '{topic}'. Try: space, animals, history, food, math."
    )


#LangGraph State 
class ChatbotState(TypedDict):
    messages: Annotated[list, add_messages]


#Bind tools to the existing model
custom_tools = [movie_recommender, fun_fact]
custom_model = model.bind_tools(custom_tools)


#Chatbot Node
def custom_chatbot_node(state: ChatbotState):
    response = custom_model.invoke(state["messages"])
    return {"messages": [response]}


#Tool Node
custom_tool_node = ToolNode(custom_tools)


#Build the Graph
custom_graph_builder = StateGraph(ChatbotState)

custom_graph_builder.add_node("chatbot", custom_chatbot_node)
custom_graph_builder.add_node("tools", custom_tool_node)

custom_graph_builder.add_edge(START, "chatbot")
custom_graph_builder.add_conditional_edges("chatbot", tools_condition)
custom_graph_builder.add_edge("tools", "chatbot")

custom_graph = custom_graph_builder.compile()


#Run the graph test
custom_result = custom_graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "Can you recommend a sci-fi movie and give me a fun fact about space?"}
        ]
    }
)

print(custom_result["messages"][-1].content)


I recommend the sci-fi movie **Arrival**, which is a thoughtful, mind-bending film that's more about language than explosions.  

Here's a fun fact about space: *A day on Venus is longer than a year on Venus — it rotates so slowly that it completes an orbit around the Sun before finishing one full spin.*


# Notes:
#
# - LangChain is the easier, higher-level framework for building agents.
# - LangGraph is the lower-level orchestration framework for building
#   stateful workflows with nodes, edges, and shared state.
# - ReAct means the model can reason, use a tool, observe the result,
#   and continue until it can answer.